# 🚀 Treinamento ResNet50 com freeze parcial, augmentação refinada e early stopping ajustado

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm
from sklearn.metrics import classification_report
import numpy as np

In [4]:
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
PATIENCE = 10
REGIONS = ["forehead", "chin", "nose", "left_cheek", "right_cheek"]
BASE_PATH = "../../data/final"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
model_dir = os.path.join(parent_dir, "models")
os.makedirs(model_dir, exist_ok=True)

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, input, target):
        ce_loss = self.ce(input, target)
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()

def get_sampler(dataset):
    targets = [s[1] for s in dataset.samples]
    class_sample_count = np.array([np.sum(np.array(targets) == t) for t in np.unique(targets)])
    weight = 1. / class_sample_count
    samples_weight = np.array([weight[t] for t in targets])
    samples_weight = torch.from_numpy(samples_weight).double()
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight), replacement=True)
    return sampler

In [5]:
for region in REGIONS:
    print(f"\n🌍 Região: {region}")
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor()
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_ds = datasets.ImageFolder(os.path.join(BASE_PATH, region, 'train'),
                                    transform=transform,
                                    is_valid_file=lambda path: "_fallback" not in path)
    val_ds = datasets.ImageFolder(os.path.join(BASE_PATH, region, 'val'),
                                  transform=transform,
                                  is_valid_file=lambda path: "_fallback" not in path)
    test_ds = datasets.ImageFolder(os.path.join(BASE_PATH, region, 'test'),
                                   transform=test_transform,
                                   is_valid_file=lambda path: "_fallback" not in path)

    sampler = get_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for name, param in model.named_parameters():
        if "layer4" not in name and "fc" not in name:
            param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 4)
    model = model.to(device)

    criterion = FocalLoss(gamma=1.5)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc=f"{region} | Epoch {epoch+1}/{EPOCHS}")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            pbar.set_postfix({'Loss': running_loss / (total / BATCH_SIZE), 'Acc': 100. * correct / total})

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total
        print(f"🧪 Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_path = os.path.join(model_dir, f"resnet50_{region}_best.pth")
            torch.save(model.state_dict(), best_path)
            print(f"💾 Novo melhor modelo salvo: {best_path}")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"⏳ EarlyStopping: {patience_counter}/{PATIENCE}")
            if patience_counter >= PATIENCE:
                print("⛔ Early stopping ativado.")
                break

    model.load_state_dict(torch.load(best_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print(f"\n📊 Classificação final para região: {region}")
    print(classification_report(all_labels, all_preds, target_names=['0', '1', '2', '3']))



🌍 Região: forehead


FileNotFoundError: Found no valid file for the classes 3. 